# Case Study 5: Spectral Anomaly Detection with a Keras Autoencoder
Train the autoencoder on normal spectra only, then flag high reconstruction-error samples.

<a href="https://colab.research.google.com/github/khairuladib94/dl-food-research/blob/main/session-6/notebooks/05_autoencoder_anomaly_tensorflow_keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

> **Colab workflow:** Save a copy in Drive, then choose Runtime > Run all. The required teaching dataset downloads automatically; Google Drive is not mounted.



## Learning route

An autoencoder learns to reconstruct normal spectra through a narrow bottleneck. A spectrum is flagged when its reconstruction error is unusually high.

![From food data to an evaluated model](../assets/session6-model-workflow.png)

The crucial difference from supervised classification is that model fitting uses only normal samples; anomaly labels are reserved for evaluation.


## 1. Prepare the notebook environment

We begin with the libraries used throughout the case study. Setting TensorFlow's log level before importing TensorFlow keeps the workshop output focused on the results.


In [ ]:
from pathlib import Path
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


### Make the data portable

The next cell first looks for `anomaly_spectra.npz` beside the notebook. If the file is absent—as it will be in a fresh Colab session—it downloads the public workshop copy. This keeps the notebook runnable without mounting Google Drive.


In [ ]:
# Portable workshop data setup: local Jupyter first, Colab fallback.
DATA_FILE = 'anomaly_spectra.npz'
LOCAL_DATA_DIR = Path('../data')
if (LOCAL_DATA_DIR / DATA_FILE).exists():
    DATA_DIR = LOCAL_DATA_DIR
    IN_COLAB = False
else:
    from urllib.request import urlretrieve
    try:
        import google.colab  # type: ignore  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
    DATA_DIR = Path('/content/dl-food-research-data') if IN_COLAB else Path.cwd() / '.workshop-data'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    target = DATA_DIR / DATA_FILE
    if not target.exists():
        url = 'https://raw.githubusercontent.com/khairuladib94/dl-food-research/main' + '/session-6/data/' + DATA_FILE
        print(f'Downloading {DATA_FILE} ...')
        urlretrieve(url, target)
    print(f'Dataset ready: {target}')


### Make the experiment repeatable

Fixed random seeds make the teaching run easier to reproduce. The helper functions also keep splitting, standardisation, and training-history summaries consistent across the five examples.

> **Important:** standardisation statistics are calculated from the training set only. Using the test set here would leak information into the model.


In [ ]:
SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)

def make_split(n, test_fraction=0.2):
    idx = np.arange(n)
    np.random.shuffle(idx)
    cut = int(n * (1 - test_fraction))
    return idx[:cut], idx[cut:]

def standardize(train, test, axis=0):
    mean = train.mean(axis=axis, keepdims=True)
    std = train.std(axis=axis, keepdims=True) + 1e-8
    return (train - mean) / std, (test - mean) / std, mean, std

def history_table(history):
    hist = pd.DataFrame(history.history)
    return hist.tail(5).round(4)


### Set up reusable plots

These helpers turn Keras histories and classification results into readable diagnostics. They do not affect training; they only help us inspect what the model learned.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (8, 4.8),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10
})

def plot_history(history, metrics=None, title='Training history'):
    hist = pd.DataFrame(history.history)
    metrics = metrics or [c for c in hist.columns if not c.startswith('val_')]
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 4))
    if len(metrics) == 1:
        axes = [axes]
    for ax, metric in zip(axes, metrics):
        ax.plot(hist.index + 1, hist[metric], label=f'train {metric}', linewidth=2)
        val_metric = f'val_{metric}'
        if val_metric in hist:
            ax.plot(hist.index + 1, hist[val_metric], label=f'validation {metric}', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend()
    fig.suptitle(title, y=1.03, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_binary_confusion(y_true, y_pred, labels, title='Confusion matrix'):
    cm = confusion_matrix(y_true.astype(int), y_pred.astype(int))
    disp = ConfusionMatrixDisplay(cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()
    return cm


## 2. Load spectra and identify normal samples

The labels are used to construct a normal-only training set and a mixed test set.


In [ ]:
data = np.load(DATA_DIR / 'anomaly_spectra.npz')
spectra = data['spectra'].astype('float32')
labels = data['labels'].astype(int)

normal_idx = np.where(labels == 0)[0]
anom_idx = np.where(labels == 1)[0]
np.random.shuffle(normal_idx)
train_normal = normal_idx[:430]
test_idx = np.concatenate([normal_idx[430:], anom_idx])


### Build normal-only training and mixed test sets

The autoencoder learns the normal pattern. Standardisation still uses only the normal training subset.


In [ ]:
X_train_raw = spectra[train_normal]
X_test_raw = spectra[test_idx]
y_test = labels[test_idx]

X_train, X_test, mean, std = standardize(X_train_raw, X_test_raw, axis=0)
print('Normal-only training:', X_train.shape, 'Mixed test:', X_test.shape)
pd.Series(y_test).value_counts().rename(index={0: 'normal', 1: 'anomaly'})


### Compare normal and anomalous spectra

Visual differences provide intuition, but the model will use reconstruction error across all bands.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cls, color, name in [(0, '#0B2A55', 'normal'), (1, '#D1495B', 'anomaly')]:
    cls_idx = np.where(labels == cls)[0][:8]
    ax.plot(spectra[cls_idx].T, color=color, alpha=0.20)
    ax.plot(spectra[cls_idx].mean(axis=0), color=color, linewidth=2.5, label=f'{name} mean')
ax.set_title('Spectral anomaly preview')
ax.set_xlabel('Spectral band index')
ax.set_ylabel('Signal intensity')
ax.legend()
plt.tight_layout()
plt.show()


## 3. Build the encoder

The encoder compresses each spectrum into a six-value bottleneck representation.


In [ ]:
input_dim = X_train.shape[1]
encoder = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(96, activation='relu'),
    layers.Dense(24, activation='relu'),
    layers.Dense(6, activation='linear', name='bottleneck')
], name='encoder')


### Build the decoder

The decoder expands the bottleneck back to the original number of spectral bands.


In [ ]:
decoder = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(24, activation='relu'),
    layers.Dense(96, activation='relu'),
    layers.Dense(input_dim)
], name='decoder')


### Connect and compile the autoencoder

Training minimizes reconstruction mean squared error between each normal input and its output.


In [ ]:
inputs = keras.Input(shape=(input_dim,))
reconstructions = decoder(encoder(inputs))
autoencoder = keras.Model(inputs, reconstructions, name='spectral_autoencoder')
autoencoder.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse')
autoencoder.summary()


## 4. Train on normal spectra

Both input and target are `X_train` because reconstruction—not a class label—is the objective.


In [ ]:
history = autoencoder.fit(
    X_train, X_train,
    validation_split=0.20,
    epochs=30,
    batch_size=32,
    verbose=0
)
history_table(history)


In [ ]:
plot_history(history, metrics=['loss'], title='Autoencoder reconstruction learning curve')


## 5. Calculate reconstruction errors

Mean squared reconstruction error summarizes how poorly each spectrum matches the learned normal pattern.


In [ ]:
train_recon = autoencoder.predict(X_train, verbose=0)
test_recon = autoencoder.predict(X_test, verbose=0)

train_err = np.mean((train_recon - X_train) ** 2, axis=1)
test_err = np.mean((test_recon - X_test) ** 2, axis=1)


### Set an anomaly threshold

The 95th percentile of normal training error is a transparent teaching rule. Real studies should tune it on separate validation data.


In [ ]:
threshold = np.quantile(train_err, 0.95)
pred_anomaly = (test_err > threshold).astype(int)


### Inspect flagged samples

The table shows how reconstruction error becomes a binary anomaly decision.


In [ ]:
print(f'Reconstruction threshold: {threshold:.5f}')
print(f'Detection accuracy: {(pred_anomaly == y_test).mean():.3f}')
pd.DataFrame({
    'true_anomaly': y_test[:15],
    'reconstruction_error': test_err[:15],
    'flagged': pred_anomaly[:15]
}).round(5)


### Count detection errors

The confusion matrix reports false alarms and missed anomalies at the chosen threshold.


In [ ]:
plot_binary_confusion(y_test, pred_anomaly, labels=['normal', 'anomaly'], title='Autoencoder anomaly confusion matrix')


### Understand the threshold and reconstruction

Error distributions show class overlap; input/reconstruction pairs reveal what the autoencoder fails to reproduce.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].hist(train_err, bins=24, alpha=0.75, label='train normal', color='#0B2A55')
axes[0].hist(test_err[y_test == 0], bins=24, alpha=0.55, label='test normal', color='#3E7CB1')
axes[0].hist(test_err[y_test == 1], bins=24, alpha=0.65, label='test anomaly', color='#D1495B')
axes[0].axvline(threshold, color='black', linestyle='--', linewidth=1.8, label='threshold')
axes[0].set_title('Reconstruction error distribution')
axes[0].set_xlabel('Mean squared reconstruction error')
axes[0].set_ylabel('Samples')
axes[0].legend(fontsize=8)

normal_example = np.where(y_test == 0)[0][0]
anomaly_example = np.where(y_test == 1)[0][0]
for idx, name, color in [(normal_example, 'normal', '#0B2A55'), (anomaly_example, 'anomaly', '#D1495B')]:
    axes[1].plot(X_test[idx], color=color, linewidth=2, label=f'{name} input')
    axes[1].plot(test_recon[idx], color=color, linestyle='--', linewidth=1.8, label=f'{name} reconstruction')
axes[1].set_title('Input vs autoencoder reconstruction')
axes[1].set_xlabel('Standardized spectral band')
axes[1].set_ylabel('Signal')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## How to read the evidence

![Three complementary views of model quality](../assets/session6-evaluation-views.png)

Learning curves describe optimization, predictions show sample-level behaviour, and error summaries expose the kinds of mistakes being made. A publishable conclusion needs all three, plus external validation that matches the intended use.


## Teaching notes
- Threshold selection should be justified using normal validation samples.
- Anomaly detection flags unusual patterns; it does not identify the root cause by itself.
- Pair this with confirmatory chemistry or expert review.